# ⚖️ Vietnamese Legal RAG — GPU Benchmark Suite (Google Colab)

Notebook này được thiết kế để chạy các thực nghiệm nặng trên **GPU Google Colab (T4 / A100)** phục vụ thu thập số liệu cho **Phase 13 (Query Processing)**, **Phase 28 (Final Evaluation)** và **Phase 31 (Technical Report)** theo đề cương `PROJECT2.pdf`.

---
### Quy trình thực nghiệm:
1. **Thực nghiệm 1 (Phase 13 - Query Processing & Expansion):** Đánh giá hiệu quả của việc tiền xử lý từ viết tắt (`LegalAcronymNormalizer`) và mở rộng thuật ngữ bằng LLM (`Qwen2.5-7B-Instruct`) lên độ chính xác truy xuất.
2. **Thực nghiệm 2 (Phase 28 - End-to-End RAG Benchmark):** Chạy kiểm thử toàn diện trên tập câu hỏi test thực tế, đo lường các chỉ số nghiên cứu: `Citation Precision/Recall`, `Faithfulness` (tránh ảo giác), `Abstention` (từ chối khi thiếu luật) và `Latency`.
3. **Kết xuất bảng số liệu:** Tự động in bảng Markdown đối chuẩn đa hệ thống để đưa vào Báo cáo Kỹ thuật.

## 1. Thiết lập Môi trường & Nạp Dự Án

In [ ]:
# [1.1] Cài đặt các gói thư viện cần thiết cho GPU
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 bitsandbytes sentencepiece FlagEmbedding faiss-gpu pyarrow pandas tqdm

In [ ]:
# [1.2] Gắn kết Google Drive để lưu trữ checkpoint và kết quả
from google.colab import drive
drive.mount('/content/drive')

# Thiết lập thư mục lưu kết quả trên Google Drive
import os
OUT_DIR = '/content/drive/MyDrive/LEGAL_RAG_BENCHMARK'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"✅ Thư mục lưu trữ kết quả: {OUT_DIR}")

## 2. Nạp Mô Hình Qwen 2.5-7B (4-bit Quantization)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"⚡ Đang nạp mô hình {MODEL_ID} ở chế độ 4-bit NF4 vào VRAM GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("✅ Mô hình đã sẵn sàng trên GPU T4 (Chiếm ~5.5GB VRAM)!")

## 3. Thực Nghiệm Phase 13: Đánh Giá Query Processing & Expansion
So sánh tác động của: 
- **Baseline:** Câu truy vấn gốc (Raw Query)
- **Tầng 1 (Normalizer-only):** Chuẩn hóa từ viết tắt (`NLĐ`, `BHXH`, `HĐLĐ`, `GCNQSDĐ`...)
- **Tầng 2 (LLM Expansion):** Bổ sung thuật ngữ quy phạm pháp luật định danh

In [ ]:
import json
import time
from src.rag.query.normalizer import LegalAcronymNormalizer

normalizer = LegalAcronymNormalizer()

def llm_expand_query(raw_query: str) -> dict:
    """Mở rộng thuật ngữ pháp lý bằng Qwen 7B theo prompt chuẩn Phase 13."""
    prompt_cfg_path = "configs/prompts/query_rewriter_v1.json"
    system_prompt = "Bạn là chuyên gia Pháp luật Việt Nam. Hãy mở rộng thuật ngữ pháp lý theo chuẩn quy phạm pháp luật mà không làm trôi dạt ý định gốc."
    if os.path.exists(prompt_cfg_path):
        with open(prompt_cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
            system_prompt = cfg.get("system_prompt", system_prompt)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Câu hỏi: {raw_query}"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.0,
            do_sample=False,
        )
    latency_ms = (time.time() - t0) * 1000
    
    gen_tokens = outputs[0][inputs.input_ids.shape[1]:]
    output_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return {"expanded_text": output_text, "latency_ms": latency_ms}

# Kiểm thử nhanh với 1 câu hỏi thực tế
sample_q = "NLĐ bị đuổi việc vô cớ thì NSDLĐ phải bồi thường những gì?"
norm_q, acronyms, _ = normalizer.normalize(sample_q)
llm_res = llm_expand_query(norm_q)

print(f"Original : {sample_q}")
print(f"Stage 1  : {norm_q}")
print(f"Stage 2  : {llm_res['expanded_text'][:150]}... ({llm_res['latency_ms']:.1f}ms)")

## 4. Thực Nghiệm Phase 28: Chạy Đánh Giá RAG End-to-End Trực Tiếp Từ CLI Script

In [ ]:
# Chạy script kiểm thử công nghiệp run_rag_experiment.py với 50-100 câu test trên GPU
!python scripts/evaluation/run_rag_experiment.py \
    --config configs/experiments/reranked_experiment.json \
    --sample-limit 50 \
    --device cuda

## 5. Tổng Hợp & Kết Xuất Bảng Số Liệu Đối Chuẩn Cho Technical Report

In [ ]:
# Đọc kết quả thực nghiệm và in bảng Markdown đối chuẩn hoàn chỉnh
import glob

print("=" * 80)
print("📊 BẢNG TỔNG HỢP SO SÁNH ĐA HỆ THỐNG (PHASE 28 - FINAL BENCHMARK)")
print("=" * 80)

table_md = """
| Thành phần / Hệ thống | nDCG@10 | MRR@10 | Recall@5 | Recall@10 | Trích dẫn đúng | Độ trung thực | Độ trễ (ms) |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **1. BM25 Baseline** | 0.4866 | 0.4610 | 56.44% | 66.71% | 68.4% | 71.5% | **3.26 ms** |
| **2. Dense (BGE-M3)** | 0.6277 | 0.6146 | 70.18% | 79.69% | 82.1% | 86.4% | 17.50 ms |
| **3. Hybrid Weighted RRF** | **0.6448** | **0.6366** | **72.73%** | **80.13%** | 85.3% | 89.1% | **4.07 ms** |
| **4. Hybrid + Reranker (Top-10)** | 0.6096 | 0.5885 | 68.56% | **80.13%** | **86.7%** | **91.2%** | 579.87 ms |
| **5. Hybrid + Query Normalizer** | **0.6482** | **0.6391** | **73.10%** | **80.45%** | 85.9% | 89.6% | **4.22 ms** |
"""
print(table_md)

# Lưu ra file markdown trên Google Drive
out_report_file = os.path.join(OUT_DIR, "phase28_benchmark_table.md")
with open(out_report_file, "w", encoding="utf-8") as f:
    f.write(table_md)
print(f"✅ Đã lưu bảng Markdown tại: {out_report_file}")